In [ ]:
# Install PySpark (ONLY if using Google Colab)
!pip install pyspark

In [ ]:
# Import SparkSession
from pyspark.sql import SparkSession

# Create Spark Session
spark = SparkSession.builder \
    .appName("RetailX Project") \
    .getOrCreate()

# Check if Spark is working
spark

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving sales_data.csv to sales_data (1).csv


In [ ]:
sales_rdd = spark.sparkContext.textFile("sales_data.csv")
sales_rdd.take(5)

['transaction_id,customer_id,product_id,store_id,quantity,price,timestamp',
 '1,101,201,1,2,500,2026-01-01',
 '2,102,202,1,1,700,2026-01-02',
 '3,103,203,2,3,300,2026-01-03',
 '4,104,204,2,2,400,2026-01-04']

In [ ]:
# Get header (first row)
header = sales_rdd.first()

# Remove header
sales_rdd_clean = sales_rdd.filter(lambda row: row != header)

# Check result
sales_rdd_clean.take(5)

['1,101,201,1,2,500,2026-01-01',
 '2,102,202,1,1,700,2026-01-02',
 '3,103,203,2,3,300,2026-01-03',
 '4,104,204,2,2,400,2026-01-04',
 '5,105,205,3,1,900,2026-01-05']

In [ ]:
# Split each row by comma
sales_split = sales_rdd_clean.map(lambda row: row.split(","))

# Check output
sales_split.take(5)

[['1', '101', '201', '1', '2', '500', '2026-01-01'],
 ['2', '102', '202', '1', '1', '700', '2026-01-02'],
 ['3', '103', '203', '2', '3', '300', '2026-01-03'],
 ['4', '104', '204', '2', '2', '400', '2026-01-04'],
 ['5', '105', '205', '3', '1', '900', '2026-01-05']]

In [ ]:
sales_final = sales_split.map(lambda x: (
    int(x[0]),   # transaction_id
    int(x[1]),   # customer_id
    int(x[2]),   # product_id
    int(x[3]),   # store_id
    int(x[4]),   # quantity
    float(x[5]), # price
    x[6]         # timestamp
))

# Check result
sales_final.take(5)

[(1, 101, 201, 1, 2, 500.0, '2026-01-01'),
 (2, 102, 202, 1, 1, 700.0, '2026-01-02'),
 (3, 103, 203, 2, 3, 300.0, '2026-01-03'),
 (4, 104, 204, 2, 2, 400.0, '2026-01-04'),
 (5, 105, 205, 3, 1, 900.0, '2026-01-05')]

In [ ]:
# STEP 5A: Map (product_id, revenue)
product_sales = sales_final.map(lambda x: (x[2], x[4] * x[5]))

product_sales.take(5)

[(201, 1000.0), (202, 700.0), (203, 900.0), (204, 800.0), (205, 900.0)]

In [ ]:
# STEP 5B: ReduceByKey
total_sales_per_product = product_sales.reduceByKey(lambda a, b: a + b)

total_sales_per_product.collect()

[(202, 2100.0),
 (204, 2800.0),
 (206, 1800.0),
 (208, 1000.0),
 (210, 1500.0),
 (201, 3000.0),
 (203, 1200.0),
 (205, 2700.0),
 (207, 3200.0),
 (209, 6000.0)]

In [ ]:
# Remove invalid records
valid_sales = sales_final.filter(lambda x: x[4] > 0 and x[5] > 0)

valid_sales.take(5)

[(1, 101, 201, 1, 2, 500.0, '2026-01-01'),
 (2, 102, 202, 1, 1, 700.0, '2026-01-02'),
 (3, 103, 203, 2, 3, 300.0, '2026-01-03'),
 (4, 104, 204, 2, 2, 400.0, '2026-01-04'),
 (5, 105, 205, 3, 1, 900.0, '2026-01-05')]

In [ ]:
# High value transactions (revenue > 1000)
high_value_sales = valid_sales.filter(lambda x: x[4] * x[5] > 1000)

high_value_sales.collect()

[(6, 101, 201, 1, 4, 500.0, '2026-01-06'),
 (7, 102, 202, 1, 2, 700.0, '2026-01-07'),
 (9, 104, 204, 2, 5, 400.0, '2026-01-09'),
 (10, 105, 205, 3, 2, 900.0, '2026-01-10'),
 (12, 107, 207, 2, 3, 800.0, '2026-01-12'),
 (14, 109, 209, 1, 4, 1000.0, '2026-01-14'),
 (16, 106, 206, 1, 2, 600.0, '2026-01-16'),
 (19, 109, 209, 1, 2, 1000.0, '2026-01-19')]

In [ ]:
# Convert RDD to DataFrame
columns = ["transaction_id", "customer_id", "product_id", "store_id", "quantity", "price", "timestamp"]

sales_df = spark.createDataFrame(sales_final, columns)

# Show data
sales_df.show(5)

+--------------+-----------+----------+--------+--------+-----+----------+
|transaction_id|customer_id|product_id|store_id|quantity|price| timestamp|
+--------------+-----------+----------+--------+--------+-----+----------+
|             1|        101|       201|       1|       2|500.0|2026-01-01|
|             2|        102|       202|       1|       1|700.0|2026-01-02|
|             3|        103|       203|       2|       3|300.0|2026-01-03|
|             4|        104|       204|       2|       2|400.0|2026-01-04|
|             5|        105|       205|       3|       1|900.0|2026-01-05|
+--------------+-----------+----------+--------+--------+-----+----------+
only showing top 5 rows


In [ ]:
# High value transactions (revenue > 1000)
high_value_df = sales_df.filter((sales_df.quantity * sales_df.price) > 1000)

high_value_df.show()

+--------------+-----------+----------+--------+--------+------+----------+
|transaction_id|customer_id|product_id|store_id|quantity| price| timestamp|
+--------------+-----------+----------+--------+--------+------+----------+
|             6|        101|       201|       1|       4| 500.0|2026-01-06|
|             7|        102|       202|       1|       2| 700.0|2026-01-07|
|             9|        104|       204|       2|       5| 400.0|2026-01-09|
|            10|        105|       205|       3|       2| 900.0|2026-01-10|
|            12|        107|       207|       2|       3| 800.0|2026-01-12|
|            14|        109|       209|       1|       4|1000.0|2026-01-14|
|            16|        106|       206|       1|       2| 600.0|2026-01-16|
|            19|        109|       209|       1|       2|1000.0|2026-01-19|
+--------------+-----------+----------+--------+--------+------+----------+



In [ ]:
# Example: quantity > 1
filtered_df = sales_df.filter(sales_df.quantity > 1)

filtered_df.show()

+--------------+-----------+----------+--------+--------+------+----------+
|transaction_id|customer_id|product_id|store_id|quantity| price| timestamp|
+--------------+-----------+----------+--------+--------+------+----------+
|             1|        101|       201|       1|       2| 500.0|2026-01-01|
|             3|        103|       203|       2|       3| 300.0|2026-01-03|
|             4|        104|       204|       2|       2| 400.0|2026-01-04|
|             6|        101|       201|       1|       4| 500.0|2026-01-06|
|             7|        102|       202|       1|       2| 700.0|2026-01-07|
|             9|        104|       204|       2|       5| 400.0|2026-01-09|
|            10|        105|       205|       3|       2| 900.0|2026-01-10|
|            12|        107|       207|       2|       3| 800.0|2026-01-12|
|            13|        108|       208|       3|       2| 200.0|2026-01-13|
|            14|        109|       209|       1|       4|1000.0|2026-01-14|
|           

In [ ]:
from pyspark.sql.functions import col, sum

# Create revenue column
sales_with_revenue = sales_df.withColumn("revenue", col("quantity") * col("price"))

# Group by store_id
revenue_per_store = sales_with_revenue.groupBy("store_id").agg(
    sum("revenue").alias("total_revenue")
)

revenue_per_store.show()

+--------+-------------+
|store_id|total_revenue|
+--------+-------------+
|       1|      12900.0|
|       3|       3700.0|
|       2|       8700.0|
+--------+-------------+



In [ ]:
revenue_per_product = sales_with_revenue.groupBy("product_id").agg(
    sum("revenue").alias("total_revenue")
)

revenue_per_product.show()

+----------+-------------+
|product_id|total_revenue|
+----------+-------------+
|       202|       2100.0|
|       201|       3000.0|
|       203|       1200.0|
|       205|       2700.0|
|       204|       2800.0|
|       209|       6000.0|
|       208|       1000.0|
|       207|       3200.0|
|       210|       1500.0|
|       206|       1800.0|
+----------+-------------+



In [ ]:
from google.colab import files
uploaded = files.upload()

Saving customer_data.csv to customer_data.csv
Saving product_data.csv to product_data.csv


In [ ]:
# Load customer data
customer_df = spark.read.csv("customer_data.csv", header=True, inferSchema=True)

# Load product data
product_df = spark.read.csv("product_data.csv", header=True, inferSchema=True)

# Check data
customer_df.show(5)
product_df.show(5)

+-----------+-----+---------+-------+
|customer_id| name|     city|segment|
+-----------+-----+---------+-------+
|        101| Amit|    Delhi|   Gold|
|        102| Neha|   Mumbai| Silver|
|        103|Rahul|Bangalore|   Gold|
|        104|Sneha|    Delhi| Bronze|
|        105|Vikas|     Pune| Silver|
+-----------+-----+---------+-------+
only showing top 5 rows
+----------+-----------+-------+
|product_id|   category|  brand|
+----------+-----------+-------+
|       201|Electronics|   Sony|
|       202|   Clothing|   Zara|
|       203|    Grocery| Nestle|
|       204|Electronics|Samsung|
|       205|   Clothing|    H&M|
+----------+-----------+-------+
only showing top 5 rows


In [ ]:
# Join sales with customer data
sales_customer_df = sales_df.join(customer_df, on="customer_id", how="inner")

sales_customer_df.show(5)

+-----------+--------------+----------+--------+--------+-----+----------+-----+---------+-------+
|customer_id|transaction_id|product_id|store_id|quantity|price| timestamp| name|     city|segment|
+-----------+--------------+----------+--------+--------+-----+----------+-----+---------+-------+
|        101|             1|       201|       1|       2|500.0|2026-01-01| Amit|    Delhi|   Gold|
|        102|             2|       202|       1|       1|700.0|2026-01-02| Neha|   Mumbai| Silver|
|        103|             3|       203|       2|       3|300.0|2026-01-03|Rahul|Bangalore|   Gold|
|        104|             4|       204|       2|       2|400.0|2026-01-04|Sneha|    Delhi| Bronze|
|        105|             5|       205|       3|       1|900.0|2026-01-05|Vikas|     Pune| Silver|
+-----------+--------------+----------+--------+--------+-----+----------+-----+---------+-------+
only showing top 5 rows


In [ ]:
# Join with product data
final_df = sales_customer_df.join(product_df, on="product_id", how="inner")

final_df.show(5)

+----------+-----------+--------------+--------+--------+-----+----------+-----+---------+-------+-----------+-------+
|product_id|customer_id|transaction_id|store_id|quantity|price| timestamp| name|     city|segment|   category|  brand|
+----------+-----------+--------------+--------+--------+-----+----------+-----+---------+-------+-----------+-------+
|       201|        101|             1|       1|       2|500.0|2026-01-01| Amit|    Delhi|   Gold|Electronics|   Sony|
|       202|        102|             2|       1|       1|700.0|2026-01-02| Neha|   Mumbai| Silver|   Clothing|   Zara|
|       203|        103|             3|       2|       3|300.0|2026-01-03|Rahul|Bangalore|   Gold|    Grocery| Nestle|
|       204|        104|             4|       2|       2|400.0|2026-01-04|Sneha|    Delhi| Bronze|Electronics|Samsung|
|       205|        105|             5|       3|       1|900.0|2026-01-05|Vikas|     Pune| Silver|   Clothing|    H&M|
+----------+-----------+--------------+--------+

In [ ]:
# Create temporary SQL table
final_df.createOrReplaceTempView("retail_data")

In [ ]:
spark.sql("SELECT * FROM retail_data").show()

+----------+-----------+--------------+--------+--------+------+----------+-----+---------+-------+-----------+-------+
|product_id|customer_id|transaction_id|store_id|quantity| price| timestamp| name|     city|segment|   category|  brand|
+----------+-----------+--------------+--------+--------+------+----------+-----+---------+-------+-----------+-------+
|       201|        101|             1|       1|       2| 500.0|2026-01-01| Amit|    Delhi|   Gold|Electronics|   Sony|
|       202|        102|             2|       1|       1| 700.0|2026-01-02| Neha|   Mumbai| Silver|   Clothing|   Zara|
|       203|        103|             3|       2|       3| 300.0|2026-01-03|Rahul|Bangalore|   Gold|    Grocery| Nestle|
|       204|        104|             4|       2|       2| 400.0|2026-01-04|Sneha|    Delhi| Bronze|Electronics|Samsung|
|       205|        105|             5|       3|       1| 900.0|2026-01-05|Vikas|     Pune| Silver|   Clothing|    H&M|
|       201|        101|             6| 

In [ ]:
spark.sql("""
SELECT
    product_id,
    SUM(quantity * price) AS total_revenue
FROM retail_data
GROUP BY product_id
ORDER BY total_revenue DESC
LIMIT 5
""").show()

+----------+-------------+
|product_id|total_revenue|
+----------+-------------+
|       209|       6000.0|
|       207|       3200.0|
|       201|       3000.0|
|       204|       2800.0|
|       205|       2700.0|
+----------+-------------+



In [ ]:
spark.sql("""
SELECT
    SUBSTRING(timestamp, 1, 7) AS month,
    SUM(quantity * price) AS monthly_revenue
FROM retail_data
GROUP BY month
ORDER BY month
""").show()

+-------+---------------+
|  month|monthly_revenue|
+-------+---------------+
|2026-01|        25300.0|
+-------+---------------+



In [ ]:
# Save final dataset to CSV
final_df.toPandas().to_csv("final_retail_data.csv", index=False)

In [ ]:
from google.colab import files
files.download("final_retail_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>